# Download A YouTube Song And Add It To The Index

This notebook downloads `https://www.youtube.com/watch?v=SieTiNcFcQY` as an MP3, creates/updates the artist and track metadata in SQLite, computes a CLAP embedding, and stores the vector in the local ChromaDB index.

By default the notebook writes to `./data/notebook_youtube_example.sqlite3` and `./data/notebook_chroma` so it is safe to run as an example. Change those paths if you want to add the song to your production index.

In [ ]:
from pathlib import Path
import os

YOUTUBE_URL = "https://www.youtube.com/watch?v=SieTiNcFcQY"
CACHE_DIR = Path(".songs_cache")

# Example-local persistence. Change these to your main paths if desired.
os.environ.setdefault("STREETPARADE_DB", "data/notebook_youtube_example.sqlite3")
os.environ.setdefault("STREETPARADE_CHROMA_DIR", "data/notebook_chroma")

Path(os.environ["STREETPARADE_DB"]).parent.mkdir(parents=True, exist_ok=True)
Path(os.environ["STREETPARADE_CHROMA_DIR"]).mkdir(parents=True, exist_ok=True)

print("SQLite:", os.environ["STREETPARADE_DB"])
print("Chroma:", os.environ["STREETPARADE_CHROMA_DIR"])

## Download The YouTube Audio

The downloader uses `yt-dlp` and FFmpeg extraction to cache the video as MP3. If `artist` is omitted, metadata is inferred from YouTube.

In [ ]:
from streetparade_embeddings.youtube import download_youtube_to_cache

download = download_youtube_to_cache(YOUTUBE_URL, CACHE_DIR)
download

## Create Metadata Rows

This creates/updates the artist row, upserts a track row for the YouTube URL, and records audio sample/chunk metadata.

In [ ]:
from datetime import UTC, datetime

from streetparade_embeddings.db import connect, init_db
from streetparade_embeddings.repositories import complete_track_download, create_or_update_artist, upsert_track
from streetparade_embeddings.schemas import ArtistCreate, DownloadRequest

def now():
    return datetime.now(UTC).isoformat()

init_db()

artist = create_or_update_artist(
    ArtistCreate(
        name=download.artist,
        links=[YOUTUBE_URL],
        youtube=YOUTUBE_URL,
    ),
    now,
)

download_request = DownloadRequest(
    max_tracks=1,
    track_urls=[YOUTUBE_URL],
    cache_dir=str(CACHE_DIR),
)

with connect() as conn:
    track_id = upsert_track(
        conn,
        artist_id=artist["id"],
        url=YOUTUBE_URL,
        path=str(download.path),
        downloaded=True,
        download_status="completed",
        now=now,
    )

track = complete_track_download(track_id, download.path, download_request, now)
dict(track)

## Compute And Store The Embedding

This loads the CLAP model, embeds the cached MP3, stores the vector in ChromaDB, and stores metadata/provenance in SQLite. The first run may download model weights from Hugging Face.

In [ ]:
from streetparade_embeddings.config import Device
from streetparade_embeddings.embeddings import ClapEmbeddingModel
from streetparade_embeddings.repositories import select_embedding_rows, store_track_embedding
from streetparade_embeddings.schemas import ComputeRequest

compute_request = ComputeRequest(
    artist_id=artist["id"],
    only_missing=True,
    device=Device.AUTO,
    max_tracks=1,
)

rows = [row for row in select_embedding_rows(compute_request) if row["id"] == track_id]
if not rows:
    raise RuntimeError("No track selected for embedding. It may already be indexed for this model/sampling configuration.")

model = ClapEmbeddingModel(model_name=compute_request.model_name, device=compute_request.device)
embedding = model.embed_track(
    rows[0]["path"],
    sampling_rate=compute_request.sampling_rate,
    chunk_seconds=compute_request.chunk_seconds,
    stride_seconds=compute_request.chunk_stride_seconds,
    max_chunks=compute_request.max_chunks,
)

updated_track = store_track_embedding(rows[0], embedding, compute_request, now)
dict(updated_track)

## Confirm The Song Is Searchable

Querying by `track_ids` uses the latest vector for the track and returns similarity results enriched with the SQLite `track_embeddings` metadata.

In [ ]:
from streetparade_embeddings.repositories import list_track_embeddings, similarity_search
from streetparade_embeddings.schemas import SimilaritySearchRequest

track_embeddings = list_track_embeddings(track_id, include_embedding=False)
results = similarity_search(SimilaritySearchRequest(track_ids=[track_id], n_results=5))

print("Track embeddings:")
for row in track_embeddings:
    print(row["vector_id"], row["embedding_model"], row["sampling_strategy_hash"])

print("\nSimilarity results:")
for result in results:
    te = result.get("track_embedding") or {}
    print(result["similarity"], te.get("source_url") or te.get("url"), result["vector_id"])